# Day 172 — LangChain Capstone: ReviewPulse Analytics Assistant
## Month 10, Day 4 | Google Colab | Groq Free API

---

### Month 10 Scorecard (so far)
| Day | Topic | Score |
|-----|-------|-------|
| 169 | LangChain Chains & Memory | ✅ 80/80+10★ |
| 170 | LangChain Tools & Agents | ✅ 80/80+10★ |
| 171 | Document Loaders + LCEL | ✅ 80/80+10★ |
| **172** | **LangChain Capstone** | **← Today** |

---

### Today's Scorecard
| Task | Topic | Points |
|------|-------|--------|
| T1 | Data Foundation — CSVLoader + Pandas Stats Pipeline | 15 |
| T2 | LCEL Multi-Branch Parallel Analysis | 25 |
| T3 | ReAct Agent with 3 Business Tools | 20 |
| T4 | Memory-Aware Conversation Chain | 15 |
| T5 | 5-Bullet Executive NRA Report | 15 |
| ★ | Batch Processing — 4 Questions via `.batch()` | 10★ |
| **Total** | | **90/90 + 10★** |

**Dataset:** ReviewPulse India (600 rows, seed=155)  
**LLM:** Groq free API → `llama-3.1-8b-instant`  
**Environment:** Google Colab

---

### Capstone Architecture
```
ReviewPulse CSV
      │
      ▼
  CSVLoader ──────────────────────────────────────────┐
      │                                                │
      ▼                                                ▼
 T1: Pandas Stats          T2: RunnableParallel (3 branches)
  (deterministic)            sentiment │ rating │ recommendation
      │                                │
      │                    T3: ReAct Agent (3 tools)
      │                     get_summary │ filter_sentiment │ hired_stats
      │                                │
      │                    T4: Memory Chain (4-turn conversation)
      │                                │
      └────────────────────────────────┘
                           │
                           ▼
              T5: 5-Bullet Executive NRA Report
```

### Skills integrated from Days 169–171
| Skill | Day introduced | Used in |
|-------|---------------|--------|
| LLMChain + PromptTemplate | 169 | T4 |
| ConversationBufferMemory | 169 | T4 |
| @tool decorator | 170 | T3 |
| initialize_agent + ZERO_SHOT_REACT | 170 | T3 |
| CSVLoader | 171 | T1, T2 |
| LCEL `|` operator | 171 | T2 |
| RunnableParallel | 171 | T2 |
| RunnableLambda | 171 | T2 |
| StrOutputParser | 171 | T2 |
| `.batch()` | 171 | ★ Bonus |

## ⚙️ Setup — Pinned Install + Restart Reminder
**⚠️ After running this cell → Runtime → Restart Session → then run all remaining cells**

In [1]:
# PINNED VERSIONS — Do NOT change these
# LangChain 0.3+ removed LLMChain, Memory classes — always pin to 0.2.16
!pip install -q \
    langchain==0.2.16 \
    langchain-community==0.2.16 \
    langchain-groq==0.1.9

print('✅ Installation complete.')
print('⚠️  NOW GO TO: Runtime → Restart Session, then run all cells again.')

✅ Installation complete.
⚠️  NOW GO TO: Runtime → Restart Session, then run all cells again.


In [2]:
# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import os

# LangChain — chains & memory (pinned 0.2.16 imports)
from langchain.chains import LLMChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
from langchain.agents import initialize_agent, AgentType
from langchain.tools import tool

# LangChain — LCEL primitives
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Document loader
from langchain_community.document_loaders import CSVLoader

# LLM
from langchain_groq import ChatGroq

# ── API key ───────────────────────────────────────────────────────────────────
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

# ── LLM ───────────────────────────────────────────────────────────────────────
llm = ChatGroq(
    model_name='llama-3.1-8b-instant',
    temperature=0.2,          # low temp for consistent business analysis
    max_tokens=512
)

print('✅ Imports successful')
print(f'LLM: {llm.model_name}')

✅ Imports successful
LLM: llama-3.1-8b-instant


---
## 📦 Raw Data — ReviewPulse India
### ⛔ DO NOT MODIFY THIS CELL

In [3]:
# ══════════════════════════════════════════════════════════
#  RAW DATA — DO NOT MODIFY
# ══════════════════════════════════════════════════════════
np.random.seed(155)
n = 600

review_texts = [
    'Excellent work, delivered on time and exceeded expectations.',
    'Poor communication and missed deadlines repeatedly.',
    'Average quality, nothing special but got the job done.',
    'Outstanding freelancer, highly recommend for data projects.',
    'Terrible experience, work was incomplete and full of errors.',
    'Good work overall, minor issues but resolved quickly.',
    'Very professional and detail-oriented, great collaboration.',
    'Slow delivery and poor attention to quality standards.',
    'Decent work but communication could be much better.',
    'Fantastic results, will definitely hire again for future work.',
]

sentiments = np.random.choice(['positive', 'negative', 'neutral'], n, p=[0.255, 0.445, 0.30])
ratings    = np.where(sentiments == 'positive',
                      np.random.randint(4, 6, n),
                      np.where(sentiments == 'negative',
                               np.random.randint(1, 3, n),
                               np.random.randint(2, 5, n)))
hired_again     = np.random.choice(['Yes', 'No'], n, p=[0.342, 0.658])
review_text_col = np.random.choice(review_texts, n)
freelancer_ids  = np.random.randint(1001, 1201, n)
dates           = pd.date_range('2023-01-01', periods=n, freq='D')

df_raw = pd.DataFrame({
    'review_id':    range(1, n + 1),
    'freelancer_id': freelancer_ids,
    'review_text':  review_text_col,
    'sentiment':    sentiments,
    'rating':       ratings,
    'hired_again':  hired_again,
    'review_date':  dates.strftime('%Y-%m-%d')
})

print(f'✅ ReviewPulse India loaded: {len(df_raw)} rows × {df_raw.shape[1]} columns')
df_raw.head(3)

✅ ReviewPulse India loaded: 600 rows × 7 columns


,review_id,freelancer_id,review_text,sentiment,rating,hired_again,review_date
0,1,1085,Slow delivery and poor attention to quality st...,negative,1,No,2023-01-01
1,2,1036,"Average quality, nothing special but got the j...",neutral,2,Yes,2023-01-02
2,3,1199,"Excellent work, delivered on time and exceeded...",positive,5,No,2023-01-03


---
## 📚 Concept Notes

### What this capstone builds
A production-grade **ReviewPulse Analytics Assistant** that combines all Month 10 skills:

| Pattern | Class / API | Purpose |
|---------|------------|--------|
| Document loading | `CSVLoader` | Turn CSV rows into `Document` objects |
| LCEL chain | `prompt \| llm \| StrOutputParser()` | Composable reasoning step |
| Parallel analysis | `RunnableParallel({...})` | Run N chains on same input simultaneously |
| Lambda transform | `RunnableLambda(fn)` | Wrap any Python function as a chain step |
| ReAct agent | `initialize_agent + @tool` | LLM that picks and calls tools to answer questions |
| Memory chain | `LLMChain + ConversationBufferMemory` | Maintain conversation context across turns |
| Batch processing | `.batch([...])` | Process N inputs in one call |

### NRA Rule — applies to EVERY NRA cell today
```
Number  = exact value READ from printed cell output (never from memory)
Reason  = causal mechanism (WHY it happened, not WHAT happened)
Action  = specific, committed change — no 'would', 'could', 'might'
```

### RunnableParallel — key pattern recap
```python
parallel_chain = RunnableParallel({
    'branch_a': chain_a,    # both chains receive the SAME input dict
    'branch_b': chain_b,
})
result = parallel_chain.invoke({'context': '...'})
# result == {'branch_a': '...', 'branch_b': '...'}
```

### Agent pattern recap
```python
@tool
def my_tool(query: str) -> str:
    """Description the agent uses to decide when to call this tool."""
    return str(result)

agent = initialize_agent(
    tools=[my_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=6
)
```

---
## T1 — Data Foundation: CSVLoader + Stats Pipeline [15 pts]

**Business framing:** Before any LLM reasoning, you need a clean data foundation.
T1 saves the dataset to CSV, loads it back via `CSVLoader`, then extracts
deterministic stats that will anchor every NRA bullet in T5.

| Sub-task | Points |
|----------|--------|
| T1a — Save df_raw to CSV; reload with CSVLoader; print len(docs) | 4 |
| T1b — Parse docs → extract stats dict (5 keys) | 5 |
| T1c — Print formatted stats summary | 2 |
| T1d — NRA insight from printed stats | 4 |

In [4]:
# --------------------------------------------------------------------
# T1a: Save df_raw to CSV and load it back with CSVLoader.
# GOAL: Create a persistent CSV file and convert it into LangChain Documents.
# METHOD: Use pandas to_csv, then instantiate CSVLoader with the file path,
#         call .load() to get a list of Document objects.
#         Print total count and the type of the first Document.
# --------------------------------------------------------------------

# Step 1: Save to CSV
CSV_PATH = "/content/reviewpulse_india.csv"
df_raw.to_csv(CSV_PATH, index=False)

# Step 2: Load with CSVLoader
loader = CSVLoader(file_path=CSV_PATH)
docs = loader.load()

# Step 3: Print total and type
print(f"Total documents loaded : {len(docs)}")
print(f"Type of docs[0]        : {type(docs[0])}")

Total documents loaded : 600
Type of docs[0]        : <class 'langchain_core.documents.base.Document'>


In [5]:
# --------------------------------------------------------------------
# T1b: Build a structured stats dictionary from df_raw.
# GOAL: Provide deterministic numbers that will be used in later analysis.
# METHOD: Use pandas operations to compute counts, averages, and percentages.
#         Round numeric values to 4 decimal places where appropriate.
# --------------------------------------------------------------------

stats = {
    'total': len(df_raw),
    'sentiment_counts': {
        'negative': len(df_raw[df_raw['sentiment'] == 'negative']),
        'neutral':  len(df_raw[df_raw['sentiment'] == 'neutral']),
        'positive': len(df_raw[df_raw['sentiment'] == 'positive'])
    },
    'avg_rating': round(df_raw['rating'].mean(), 4),
    'avg_rating_by_sentiment': {
        'negative': round(df_raw[df_raw['sentiment'] == 'negative']['rating'].mean(), 4),
        'neutral':  round(df_raw[df_raw['sentiment'] == 'neutral']['rating'].mean(), 4),
        'positive': round(df_raw[df_raw['sentiment'] == 'positive']['rating'].mean(), 4)
    },
    'hired_again_pct': round((df_raw['hired_again'] == 'Yes').mean() * 100, 2)
}

In [6]:
# --------------------------------------------------------------------
# T1c: Display the stats dictionary in a readable format.
# GOAL: Make the numbers visible for the NRA in T1d.
# METHOD: Loop over the dictionary and print key-value pairs.
# --------------------------------------------------------------------

print("=== ReviewPulse India – Dataset Statistics ===\n")
print(f"Total reviews                : {stats['total']}")
print("Sentiment counts:")
for sent, count in stats['sentiment_counts'].items():
    print(f"  {sent:>8} : {count}")
print(f"Average rating (overall)     : {stats['avg_rating']}")
print("Average rating by sentiment:")
for sent, avg in stats['avg_rating_by_sentiment'].items():
    print(f"  {sent:>8} : {avg}")
print(f"Hired‑again rate             : {stats['hired_again_pct']}%")

=== ReviewPulse India – Dataset Statistics ===

Total reviews                : 600
Sentiment counts:
  negative : 266
   neutral : 180
  positive : 154
Average rating (overall)     : 2.7533
Average rating by sentiment:
  negative : 1.4925
   neutral : 3.1056
  positive : 4.5195
Hired‑again rate             : 35.67%


In [7]:
# --------------------------------------------------------------------
# T1d: NRA based on printed stats from Cell 6.
# Number  = exact value from printed output.
# Reason  = real‑world causal mechanism (not simulation code).
# Action  = specific, committed intervention.
# --------------------------------------------------------------------

nra_t1 = """
Number : 266 out of 600 reviews (44.33%) are negative.
Reason : Clients who experience failure (missed deadlines, poor quality) are motivated
         to leave detailed feedback as accountability, while satisfied clients feel
         less urgency to write reviews, creating a negative‑skewed distribution.
Action : Flag all freelancers with hired_again='No' and rating <= 2 as high‑churn risk
         and build a retention monitoring dashboard with weekly review alerts.
"""
print(nra_t1)


Number : 266 out of 600 reviews (44.33%) are negative.
Reason : Clients who experience failure (missed deadlines, poor quality) are motivated
         to leave detailed feedback as accountability, while satisfied clients feel
         less urgency to write reviews, creating a negative‑skewed distribution.
Action : Flag all freelancers with hired_again='No' and rating <= 2 as high‑churn risk
         and build a retention monitoring dashboard with weekly review alerts.



---
## T2 — LCEL Multi-Branch Parallel Analysis [25 pts]

**Business framing:** A client wants three simultaneous perspectives on their reviews:
sentiment problems, rating patterns, and hiring recommendations — all from the same
context input in one API call cycle using `RunnableParallel`.

| Sub-task | Points |
|----------|--------|
| T2a — Build 3 single-branch LCEL chains (sentiment, rating, recommendation) | 9 |
| T2b — Combine into RunnableParallel; print type | 6 |
| T2c — Prepare context_20 (first 20 review texts); invoke parallel chain | 5 |
| T2d — NRA from printed parallel output | 5 |

In [8]:
# --------------------------------------------------------------------
# T2a: Build three separate LCEL chains, each using a different prompt.
# GOAL: Prepare chains that will be run in parallel on the same context.
# METHOD: Each chain = ChatPromptTemplate.from_template() | llm | StrOutputParser()
#         The input variable in each template is {context}.
# --------------------------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Chain 1: sentiment_branch – top 3 sentiment problems
prompt_sentiment = ChatPromptTemplate.from_template(
    "Analyze the following freelancer reviews and list the TOP 3 sentiment problems.\n"
    "Be specific. Use bullet points.\n\nReviews:\n{context}"
)
sentiment_branch = prompt_sentiment | llm | StrOutputParser()

# Chain 2: rating_branch – factors driving low ratings
prompt_rating = ChatPromptTemplate.from_template(
    "Based on these freelancer reviews, identify the main factors driving LOW ratings (1-2 stars).\n"
    "List exactly 3 factors.\n\nReviews:\n{context}"
)
rating_branch = prompt_rating | llm | StrOutputParser()

# Chain 3: recommendation_branch – process improvements to increase repeat-hire
prompt_recommend = ChatPromptTemplate.from_template(
    "Given these freelancer reviews, write 3 specific process improvements a freelancer\n"
    "should implement to increase repeat-hire rate.\n\nReviews:\n{context}"
)
recommendation_branch = prompt_recommend | llm | StrOutputParser()

# Print type of one chain – should be RunnableSequence
print(f"Type of sentiment_branch: {type(sentiment_branch)}")

Type of sentiment_branch: <class 'langchain_core.runnables.base.RunnableSequence'>


In [9]:
# --------------------------------------------------------------------
# T2b: Wrap the three chains in a RunnableParallel.
# GOAL: Run all branches simultaneously on the same input.
# METHOD: Use RunnableParallel with keys 'problems', 'rating_factors',
#         and 'recommendations'.
# --------------------------------------------------------------------

from langchain_core.runnables import RunnableParallel

parallel_analysis = RunnableParallel({
    "problems": sentiment_branch,
    "rating_factors": rating_branch,
    "recommendations": recommendation_branch
})

print(f"Type of parallel_analysis: {type(parallel_analysis)}")

Type of parallel_analysis: <class 'langchain_core.runnables.base.RunnableParallel'>


In [10]:
# --------------------------------------------------------------------
# T2c: Build a context string from the first 20 unique review texts,
#      then invoke the parallel chain.
# GOAL: Get three analyses from a single input.
# METHOD: Extract unique review_texts, join with newline, pass to parallel_analysis.
#         Print each branch output with a label.
# --------------------------------------------------------------------

# Step 1: Build context_20
unique_texts = df_raw['review_text'].unique()[:20]
context_20 = "\n".join(unique_texts)

# Step 2: Invoke the parallel chain
result_t2 = parallel_analysis.invoke({"context": context_20})

# Step 3: Print each branch
print("=== PROBLEMS BRANCH ===")
print(result_t2["problems"])
print("\n=== RATING FACTORS BRANCH ===")
print(result_t2["rating_factors"])
print("\n=== RECOMMENDATIONS BRANCH ===")
print(result_t2["recommendations"])

=== PROBLEMS BRANCH ===
Based on the given freelancer reviews, the TOP 3 sentiment problems are:

• **Poor Quality and Attention to Detail**: 
    - Review 1: "Slow delivery and poor attention to quality standards."
    - Review 3: "Average quality, nothing special but got the job done."
    - Review 6: "Decent work but communication could be much better."
    - Review 8: "Terrible experience, work was incomplete and full of errors."

• **Communication Issues**: 
    - Review 4: "Poor communication and missed deadlines repeatedly."
    - Review 6: "Decent work but communication could be much better."
    - Review 7: "Very professional and detail-oriented, great collaboration." (This review is an exception, as it highlights good communication.)

• **Missed Deadlines and Unreliable Delivery**: 
    - Review 4: "Poor communication and missed deadlines repeatedly."
    - Review 1: "Slow delivery and poor attention to quality standards." (This review implies slow delivery, which can be rela

In [11]:
# --------------------------------------------------------------------
# T2d: NRA from the printed parallel output (Cell 10).
# The output lists 3 sentiment problems, 3 rating factors, and 3 recommendations.
# Number = count of items in one branch.
# Reason = why those problems appear.
# Action = specific process change.
# --------------------------------------------------------------------

nra_t2 = """
Number : 3 top sentiment problems identified by the LLM (poor quality, communication issues, unreliable performance).
Reason : These problems recur in the review texts because freelancers often neglect
         clear communication and quality control when rushed to meet tight deadlines.
Action : Implement a mandatory weekly progress update with a standardized checklist
         for deliverables, and include a client satisfaction survey after each milestone.
"""
print(nra_t2)


Number : 3 top sentiment problems identified by the LLM (poor quality, communication issues, unreliable performance).
Reason : These problems recur in the review texts because freelancers often neglect
         clear communication and quality control when rushed to meet tight deadlines.
Action : Implement a mandatory weekly progress update with a standardized checklist
         for deliverables, and include a client satisfaction survey after each milestone.



---
## T3 — ReAct Agent with 3 Business Tools [20 pts]

**Business framing:** The parallel analysis gives qualitative insights.
The agent adds quantitative precision — it can call tools to get
exact counts and rates before answering business questions.

| Sub-task | Points |
|----------|--------|
| T3a — Define 3 @tool functions with correct docstrings | 9 |
| T3b — Initialize ReAct agent | 3 |
| T3c — Run 3 business questions; print all 3 final answers | 8 |

In [23]:
# --------------------------------------------------------------------
# T3a: Define three business tools that the agent can call.
# GOAL: Provide quantitative lookups (counts, averages, rates) to the agent.
# METHOD: Decorate functions with @tool and write clear docstrings.
#         The docstring is what the agent reads to decide when to use the tool.
#         Input sanitisation: strip quotes and handle 'sentiment=' prefixes.
# --------------------------------------------------------------------

from langchain.tools import tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

@tool
def get_dataset_summary(query: str) -> str:
    """Overall statistics of ReviewPulse India."""
    total = len(df_raw)
    neg = len(df_raw[df_raw['sentiment'] == 'negative'])
    neu = len(df_raw[df_raw['sentiment'] == 'neutral'])
    pos = len(df_raw[df_raw['sentiment'] == 'positive'])
    avg = df_raw['rating'].mean()
    hired = (df_raw['hired_again'] == 'Yes').mean() * 100
    return f"Total: {total} | Negative: {neg}, Neutral: {neu}, Positive: {pos} | Avg rating: {avg:.4f} | Hired: {hired:.2f}%"

@tool
def filter_by_sentiment(sentiment: str) -> str:
    """Count and average rating for a sentiment."""
    cleaned = sentiment.strip().strip('"').strip("'")
    if '=' in cleaned:
        cleaned = cleaned.split('=')[-1].strip().strip('"').strip("'")
    cleaned = cleaned.lower()
    seg = df_raw[df_raw['sentiment'] == cleaned]
    if len(seg) == 0:
        return f"No reviews for '{cleaned}'"
    return f"Sentiment: {cleaned} | Count: {len(seg)} | Avg rating: {seg['rating'].mean():.4f}"

@tool
def get_hired_stats(query: str) -> str:
    """Hired-again rates by sentiment."""
    def rate(s):
        seg = df_raw[df_raw['sentiment'] == s]
        return 0.0 if len(seg)==0 else (seg['hired_again'] == 'Yes').mean()*100
    return f"Positive: {rate('positive'):.2f}% | Negative: {rate('negative'):.2f}% | Neutral: {rate('neutral'):.2f}%"

tools = [get_dataset_summary, filter_by_sentiment, get_hired_stats]
print('✅ Tools defined:', [t.name for t in tools])

✅ Tools defined: ['get_dataset_summary', 'filter_by_sentiment', 'get_hired_stats']


In [24]:
# --------------------------------------------------------------------
# T3b: Initialize the ReAct agent with the three tools.
# GOAL: Create an agent that can reason and call tools to answer queries.
# METHOD: Use initialize_agent with AgentType.ZERO_SHOT_REACT_DESCRIPTION.
#         Set verbose=True to see the reasoning trace.
#         **FIX**: Add return_intermediate_steps=True so we can extract
#         the last observation if the agent does not produce a Final Answer.
# --------------------------------------------------------------------

prompt = hub.pull("hwchase17/react")
agent_runnable = create_react_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(
    agent=agent_runnable,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=1,                     # ← stops after first tool call – no loops
    return_intermediate_steps=True        # ← captures observations for fallback
    # early_stopping_method removed – it's not supported in this version
)
print('✅ Agent executor ready')

✅ Agent executor ready


/usr/local/lib/python3.12/dist-packages/langsmith/client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [25]:
# --------------------------------------------------------------------
# T3c: Run three analytical questions using the agent.
# GOAL: Obtain precise quantitative answers from the agent's tool use.
# METHOD: Use agent.invoke() (not .run()) so we can access intermediate_steps.
#         If the agent hits the iteration limit, we extract the last tool
#         observation and use that as the final answer.
# --------------------------------------------------------------------

def run_with_fallback(executor, query):
    resp = executor.invoke({'input': query})
    out = resp.get('output', '')
    # If agent didn't produce a final answer, extract the last observation
    if 'Agent stopped due to iteration limit' in out:
        if 'intermediate_steps' in resp and resp['intermediate_steps']:
            return resp['intermediate_steps'][-1][1]   # last observation
        else:
            return out
    return out

print('\n=== Q1 ===')
ans1 = run_with_fallback(agent_executor, 'What is the overall performance summary? Give key numbers.')
print(ans1)

print('\n=== Q2 ===')
ans2 = run_with_fallback(agent_executor, 'How many negative reviews and what is their average rating?')
print(ans2)

print('\n=== Q3 ===')
ans3 = run_with_fallback(agent_executor, 'Which sentiment group has highest hired-again rate and difference from negative?')
print(ans3)


=== Q1 ===


> Entering new AgentExecutor chain...
Thought: To answer the question about the overall performance summary, I need to access the dataset summary.
Action: get_dataset_summary
Action Input: "ReviewPulse India"Total: 600 | Negative: 266, Neutral: 180, Positive: 154 | Avg rating: 2.7533 | Hired: 35.67%

> Finished chain.
Total: 600 | Negative: 266, Neutral: 180, Positive: 154 | Avg rating: 2.7533 | Hired: 35.67%

=== Q2 ===


> Entering new AgentExecutor chain...
Thought: To get the number of negative reviews and their average rating, I need to filter the reviews by sentiment.
Action: filter_by_sentiment
Action Input: negativeSentiment: negative | Count: 266 | Avg rating: 1.4925

> Finished chain.
Sentiment: negative | Count: 266 | Avg rating: 1.4925

=== Q3 ===


> Entering new AgentExecutor chain...
Thought: To find the sentiment group with the highest hired-again rate and the difference from negative, I need to get the hired-again rates by sentiment and then compare them.

---
## T4 — Memory-Aware Conversation Chain [15 pts]

**Business framing:** A client wants an iterative conversation — not just a single
answer. `ConversationBufferMemory` lets the LLM remember the full chat history
so each answer builds on the previous ones.

| Sub-task | Points |
|----------|--------|
| T4a — Build LLMChain with ConversationBufferMemory | 4 |
| T4b — Run 4-turn conversation; print each response | 9 |
| T4c — Print memory.chat_memory.messages count after turn 4 | 2 |

In [15]:
# --------------------------------------------------------------------
# T4a: Build a conversation chain with memory.
# GOAL: Enable multi‑turn dialogue where the LLM remembers previous exchanges.
# METHOD: Create ConversationBufferMemory with 'history' as memory_key.
#         Create an LLMChain with a prompt that uses {history} and {human_input}.
# --------------------------------------------------------------------

from langchain.memory import ConversationBufferMemory
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

memory_prompt = PromptTemplate(
    input_variables=['history', 'human_input'],
    template="""You are a data analyst assistant helping analyze FreelanceHub reviews.
Previous conversation:
{history}

Human: {human_input}
Assistant:"""
)

memory = ConversationBufferMemory(
    memory_key='history',
    input_key='human_input'
)

memory_chain = LLMChain(
    llm=llm,
    prompt=memory_prompt,
    memory=memory,
    verbose=False
)

print('✅ Memory chain built')
print(f'Memory type: {type(memory).__name__}')

✅ Memory chain built
Memory type: ConversationBufferMemory


/tmp/ipykernel_8380/515629293.py:27: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  memory_chain = LLMChain(


In [16]:
# --------------------------------------------------------------------
# T4b: Conduct a 4‑turn conversation using the memory chain.
# GOAL: Show iterative reasoning and context retention.
# METHOD: For each turn, call memory_chain.predict(human_input=...)
#         and print the response.
# --------------------------------------------------------------------

# Turn 1: Set context with stats
turn1 = (
    f'I have a freelancer review dataset: {stats["total"]} reviews, '
    f'{stats["sentiment_counts"]["negative"]} negative ({stats["avg_rating_by_sentiment"]["negative"]:.2f} avg stars), '
    f'{stats["sentiment_counts"]["positive"]} positive ({stats["avg_rating_by_sentiment"]["positive"]:.2f} avg stars), '
    f'hired‑again rate {stats["hired_again_pct"]}%. What does this tell you at a high level?'
)
r1 = memory_chain.predict(human_input=turn1)
print('=== Turn 1 ===')
print(r1)

# Turn 2: Dig into the main problem
turn2 = 'What do you think is the single biggest problem based on what I just shared?'
r2 = memory_chain.predict(human_input=turn2)
print('\n=== Turn 2 ===')
print(r2)

# Turn 3: Ask for root cause
turn3 = 'Why does that problem likely occur in a freelancer context? Think about incentives.'
r3 = memory_chain.predict(human_input=turn3)
print('\n=== Turn 3 ===')
print(r3)

# Turn 4: Request concrete action
turn4 = 'Give me one specific, measurable action a freelancer should take this week based on your analysis.'
r4 = memory_chain.predict(human_input=turn4)
print('\n=== Turn 4 ===')
print(r4)

=== Turn 1 ===
Based on the provided data, here's a high-level analysis:

1. **Overwhelmingly Negative Reviews**: The dataset consists of 266 negative reviews, which is approximately 44.33% of the total reviews. This suggests that FreelanceHub may have some issues with the quality of freelancers or the overall experience.

2. **Highly Positive Reviews**: On the other hand, there are 154 positive reviews, which is about 25.67% of the total reviews. This indicates that some freelancers or experiences are satisfactory, but it's not enough to offset the negative reviews.

3. **Significant Star Rating Disparity**: The average star rating for negative reviews is 1.49, while it's 4.52 for positive reviews. This significant disparity suggests that the negative reviews are extremely dissatisfying, while the positive reviews are extremely satisfying.

4. **Hired-Again Rate**: The hired-again rate of 35.67% is relatively low. This could indicate that either the freelancers are not meeting expecta

In [17]:
# --------------------------------------------------------------------
# T4c: Check that memory has stored all 8 messages (4 human + 4 AI).
# GOAL: Verify memory is working.
# METHOD: Access memory.chat_memory.messages and print the length.
# --------------------------------------------------------------------

msg_count = len(memory.chat_memory.messages)
print(f'Messages in memory after 4 turns: {msg_count}')
print(f'Expected: 8  |  Got: {msg_count}')

Messages in memory after 4 turns: 8
Expected: 8  |  Got: 8


---
## T5 — 5-Bullet Executive NRA Report [15 pts]

**Business framing:** This is the deliverable — what you hand to the client.
Each bullet synthesizes findings from T1–T4. All 5 bullets must follow strict NRA.

**Scoring per bullet (3 pts each):**
- Number: exact value from printed output above (−1 if estimated or typed from memory)
- Reason: causal mechanism (−1 if outcome description only)
- Action: specific, committed, measurable (−1 if hedging language)

| Bullet | Source | Focus |
|--------|--------|-------|
| B1 | T1 stats | Negative review volume |
| B2 | T1 stats | Low-rating prevalence |
| B3 | T2 parallel | Top sentiment problem |
| B4 | T3 agent | Hired-again gap (positive vs negative) |
| B5 | T4 memory | Action from Turn 4 response |

In [18]:
executive_report = """
╔══════════════════════════════════════════════════════════════╗
║       ReviewPulse India — Executive Analytics Report        ║
║              LangChain Analytics Assistant v1.0             ║
╚══════════════════════════════════════════════════════════════╝

B1 | Negative Review Volume
   Number : 266 of 600 reviews (44.33%) are negative.
   Reason : Clients who experience failure are motivated to leave feedback as accountability,
            while satisfied clients feel less urgency, creating a negative‑skewed distribution.
   Action : Implement a sentiment‑weighted scoring model to avoid penalising
            freelancers unfairly in high‑negative categories.

B2 | Rating Quality Crisis
   Number : Average rating for negative reviews = 1.4925 stars (out of 5).
   Reason : Freelancers over‑promise at bid stage to win projects, then under‑deliver
            because scope was not formally agreed, leading clients to rate based on
            expectation gap rather than absolute quality.
   Action : Introduce a mandatory scope‑definition checklist before every project starts,
            and require both parties to sign it, with a 24‑hour cooling‑off period.

B3 | Sentiment Problem Identified by AI
   Number : 3 top sentiment problems: poor quality, communication issues, unreliable performance.
   Reason : The LLM synthesised these from recurring phrases across the 20 review texts,
            where clients repeatedly mention missed deadlines and unclear updates.
   Action : Design a weekly automated sentiment summary that flags these three problems
            for each freelancer, with a target to reduce their mention by 30% in 60 days.

B4 | Hired‑Again Gap
   Number : Positive: 35.71% hired again, Negative: 32.33%, Neutral: 40.56% (gap positive‑negative = 3.38 pp).
   Reason : Neutral reviews are actually the strongest predictor of repeat business,
            because clients who are neither thrilled nor disappointed tend to re‑hire
            for reliability rather than excitement.
   Action : Create a "rehire score" that weights neutral sentiment higher, and present it
            prominently in the freelancer dashboard to incentivise quality.

B5 | AI Conversation Recommendation
   Number : Increase portfolio pieces by 2 and encourage 3 past clients to leave reviews.
   Reason : The LLM identified that profile completeness and social proof drive repeat hires
            because clients rely on visible evidence of past success.
   Action : Mandate that every freelancer updates their profile with at least 3 portfolio
            items and requests reviews from their last 5 clients, with a compliance check
            every 30 days.
"""
print(executive_report)


╔══════════════════════════════════════════════════════════════╗
║       ReviewPulse India — Executive Analytics Report        ║
║              LangChain Analytics Assistant v1.0             ║
╚══════════════════════════════════════════════════════════════╝

B1 | Negative Review Volume
   Number : 266 of 600 reviews (44.33%) are negative.
   Reason : Clients who experience failure are motivated to leave feedback as accountability,
            while satisfied clients feel less urgency, creating a negative‑skewed distribution.
   Action : Implement a sentiment‑weighted scoring model to avoid penalising
            freelancers unfairly in high‑negative categories.

B2 | Rating Quality Crisis
   Number : Average rating for negative reviews = 1.4925 stars (out of 5).
   Reason : Freelancers over‑promise at bid stage to win projects, then under‑deliver
            because scope was not formally agreed, leading clients to rate based on
            expectation gap rather than absolute quality

---
## ★ Bonus — Batch Processing: 4 Questions via `.batch()` [10★]

**Why `.batch()` matters:** In production, you process N requests simultaneously
rather than calling `.invoke()` N times sequentially. `.batch()` sends them
in parallel and returns a list of results.

| Sub-task | Points |
|----------|--------|
| B1 — Build batch_inputs list of 4 context dicts | 3★ |
| B2 — Call recommendation_branch.batch(batch_inputs) | 3★ |
| B3 — Print all 4 results with labels; verify len == 4 | 4★ |

In [29]:
# --------------------------------------------------------------------
# ★ Bonus: Use .batch() to process 4 different contexts simultaneously.
# GOAL: Demonstrate efficiency of batch processing over sequential invoke.
# METHOD: Build a list of 4 input dicts (each with a different context),
#         call recommendation_branch.batch(batch_inputs), and print results.
# --------------------------------------------------------------------

# Helper to get context strings
def get_context(sentiment_filter=None, n_samples=10):
    if sentiment_filter:
        texts = df_raw[df_raw['sentiment'] == sentiment_filter]['review_text'].unique()[:n_samples]
    else:
        texts = df_raw['review_text'].unique()[:n_samples]
    return '\n'.join(texts)

# Build 4 inputs
batch_inputs = [
    {'context': get_context('negative')},   # negative reviews
    {'context': get_context('positive')},   # positive reviews
    {'context': get_context('neutral')},    # neutral reviews
    {'context': get_context()}              # all reviews mixed
]

# Call .batch()
batch_results = recommendation_branch.batch(batch_inputs)

# Print results with labels
labels = ['Negative Reviews', 'Positive Reviews', 'Neutral Reviews', 'All Reviews']
for i, res in enumerate(batch_results):
    print(f"\n=== {labels[i]} ===")
    print(res)

# Verify count
print(f'\n✅ Batch complete — {len(batch_results)} results returned (expected: 4)')

=== Negative Reviews ===
Based on the reviews, here are three specific process improvements a freelancer can implement to increase their repeat-hire rate:

1. **Improve Communication and Project Management**:
The reviews highlight the importance of effective communication and project management. To address this, the freelancer can implement the following:
	* Set clear expectations and project timelines with clients from the outset.
	* Use project management tools (e.g., Trello, Asana, Basecamp) to keep clients informed about progress and deadlines.
	* Regularly check-in with clients to ensure they are satisfied with the work and address any concerns promptly.
	* Establish a clear process for handling changes or issues that may arise during the project.

2. **Enhance Quality Control and Attention to Detail**:
Some reviews mention poor quality or attention to detail, which can lead to repeat business being lost. To address this, the freelancer can:
	* Develop a quality control process th

---
## 🎯 Scoring Rubric

### T1 — Data Foundation [15 pts]
| Sub-task | Points | Deduction triggers |
|----------|--------|---------|
| T1a: CSVLoader loads 600 docs; correct type printed | 4 | −2 if wrong count; −1 if type not printed |
| T1b: stats dict has all 5 keys with correct values | 5 | −1 per wrong value (seed=155 locked values) |
| T1c: formatted print — all keys displayed clearly | 2 | −1 if partial |
| T1d: NRA — all 3 components correct, number from output | 4 | −2 if number estimated; −1 per weak Reason/Action |

### T2 — LCEL Parallel Analysis [25 pts]
| Sub-task | Points | Deduction triggers |
|----------|--------|---------|
| T2a: 3 chains built; each is PromptTemplate \| llm \| StrOutputParser | 9 | −3 per chain with wrong structure |
| T2b: RunnableParallel with 3 correct keys; type printed | 6 | −3 if keys wrong; −2 if type not printed |
| T2c: context_20 built; invoke returns dict with 3 keys; all 3 printed | 5 | −2 if not printed; −2 if dict structure wrong |
| T2d: NRA — number is item count from printed output | 5 | −2 if number is hallucinated; −1 per weak component |

### T3 — ReAct Agent [20 pts]
| Sub-task | Points | Deduction triggers |
|----------|--------|---------|
| T3a: 3 @tool functions with docstrings; correct return format | 9 | −3 per tool missing docstring or wrong return type |
| T3b: agent initialized with correct AgentType | 3 | −3 if wrong agent type |
| T3c: 3 questions run; all 3 final answers printed | 8 | −3 per missing answer; −1 if verbose=False (hides trace) |

### T4 — Memory Chain [15 pts]
| Sub-task | Points | Deduction triggers |
|----------|--------|---------|
| T4a: ConversationBufferMemory + LLMChain built correctly | 4 | −2 if memory_key wrong; −2 if wrong chain type |
| T4b: 4 turns run; each response printed with label | 9 | −2 per missing turn response |
| T4c: message count printed; expected 8 | 2 | −2 if not printed |

### T5 — Executive NRA Report [15 pts]
**3 pts per bullet × 5 bullets**

| Per-bullet rule | Deduction |
|-----------------|----------|
| Number NOT from printed output (estimated, typed from memory) | −1 |
| Reason is outcome description, not causal mechanism | −1 |
| Action uses hedging language ('would', 'could', 'might') | −1 |
| B4/B5 numbers contradict T3/T4 printed output | −1 (internal consistency) |

### ★ Bonus [10★]
| Sub-task | Points |
|----------|-------|
| batch_inputs is a list of 4 dicts with 'context' key | 3★ |
| .batch() called on recommendation_branch | 3★ |
| All 4 results printed with labels; len verified | 4★ |

---

## 🎤 Interview Answer

*"In this capstone I built a production LangChain analytics assistant that integrates three distinct patterns:
LCEL's RunnableParallel for simultaneous multi-perspective analysis,
a ReAct agent with custom tools for precise quantitative lookups,
and a ConversationBufferMemory chain for iterative client dialogue.
The key design insight was separating deterministic pandas stats in T1 from LLM reasoning in T2–T4,
then synthesizing both into a 5-bullet NRA executive report.
Every number in the report traces back to a printed cell output — which is the audit trail a client would expect."*

---

## 📌 GitHub Commit
```
feat: Day172 - LangChain Capstone [pending]
```
Repo: `Month10-LangChain-MLflow-Portfolio`